# Atividade 1 - DataFrames

### Análise dos dados abertos da UFERSA

Nesta atividade, vamos fazer uma análise de alguns conjuntos de dados do Portal de Dados Abertos da UFERSA que estão disponíveis no endereço https://dadosabertos.ufersa.edu.br/. A solução deve ser feita utilizando apenas a API de DataFrames do Apache Spark.

#### Instruções

- A atividade pode ser feita em trio
- A entrega deve ser feita através do SIGAA enviando o notebook com solução no formato IPython
- Prazo: 20/10/2025
- Valor: 4 pontos

### Grupo
**Nome:** Lavínia Dantas de Mesquita

**Nome:** Luis Felipe Pereira de Paiva Pordeus

**Nome:** Maria Eduarda Ferreira Souza

## Atividade

**1** - Faça a ingestão dos datasets abaixo, salvando os seus arquivos no Catalog em um volume chamado `datasets` dentro de um schema chamado `atividades`:
- Quantitativos de Alunos dos Cursos de Graduação (https://api.ufersa.edu.br/apiufersa/rest/pda/estatisticas-cursos-graduacao)
- Projetos de Pesquisa (https://api.ufersa.edu.br/apiufersa/rest/pda/projetos-pesquisa)
- Bolsas de Iniciação Científica (https://api.ufersa.edu.br/apiufersa/rest/pda/bolsistas-ic)
- Disciplinas dos Cursos de Graduação (https://api.ufersa.edu.br/apiufersa/rest/pda/disciplinas-cursos-graduacao)

Você deve criar o schema e o volume no Catalog antes de fazer a ingestão.

In [0]:
%sql
-- Para facilitar a criação dentro do catalog, utilize esse bloco de código sql
-- SÓ RODE NA PRIMEIRA VEZ, SE VOCÊ NÃO TIVER CONFIGURADO
CREATE SCHEMA IF NOT EXISTS atividades;
CREATE VOLUME IF NOT EXISTS atividades.datasets;

-- Quando esse comando for executado, vá em catalog, selecione o workspace (ou seu local de trabalho escolhido, e lá já estará pronto)
-- Aí joga os arquivos lá

In [0]:
# Automatização da ingestão eu já não sei fazer, fico devendo ~Lavínia

**2** - Para cada dataset, crie um schema para definir os tipos de suas colunas e crie um DataFrame a partir da leitura do dataset no Catalog com o schema criado.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
spark = SparkSession.builder.getOrCreate()

schema_disciplinas = StructType([ # É importante lembrar de checar a ordem dos atributos antes, pra criar os schemas
    StructField("curso", StringType(), True),
    StructField("turno", StringType(), True),
    StructField("campi", StringType(), True),
    StructField("nome_curriculo", StringType(), True),
    StructField("nome_disciplina", StringType(), True),
    StructField("obrigatoria", StringType(), True),
    StructField("carga_horaria_total_disciplina", IntegerType(), True)
])

schema_bolsas = StructType([
    StructField("nomediscente", StringType(), True),
    StructField("nomeorientador", StringType(), True),
    StructField("denominacao_bolsa", StringType(), True),
    StructField("fonte_pagadora", StringType(), True),
    StructField("data_inicio_concessao_bolsa", StringType(), True),
    StructField("data_fim_concessao_bolsa", StringType(), True)
])

schema_projetos = StructType([
    StructField("codigo", StringType(), True),
    StructField("titulo_projeto", StringType(), True),
    StructField("linha_de_pesquisa", StringType(), True),
    StructField("area_conhecimento_cnpq", StringType(), True),
    StructField("unidade", StringType(), True),
    StructField("coordenador_docente_interno", StringType(), True),
    StructField("coordenador_docente_externo", StringType(), True),
    StructField("palavras_chave", StringType(), True),
    StructField("data_inicio", StringType(), True),
    StructField("data_fim", StringType(), True),
    StructField("motivo_finalizacao", StringType(), True),
    StructField("status_atual", StringType(), True),
    StructField("entidade_financiadora", StringType(), True),
    StructField("valor_financiamento", DoubleType(), True)
])

schema_estatisticas = StructType([
    StructField("ano", IntegerType(), True),
    StructField("periodo", StringType(), True),
    StructField("curso", StringType(), True),
    StructField("municipio", StringType(), True),
    StructField("turno", StringType(), True),
    StructField("ingressantes", IntegerType(), True),
    StructField("ingressantes_nao_cotistas", IntegerType(), True),
    StructField("ingressantes_cotistas", IntegerType(), True),
    StructField("ativos", IntegerType(), True),
    StructField("concluidos", IntegerType(), True),
    StructField("trancados", IntegerType(), True),
    StructField("cancelados", IntegerType(), True)
])



In [0]:
# Caminhos dos arquivos
path_estatisticas = "/Volumes/workspace/atividades/datasets/estatisticasCursosGraduacaoCSV.csv"
path_projetos = "/Volumes/workspace/atividades/datasets/projetosPesquisaCSV.csv"
path_bolsas = "/Volumes/workspace/atividades/datasets/bolsistasICCSV.csv"
path_disciplinas = "/Volumes/workspace/atividades/datasets/disciplinasCursosGraduacaoCSV.csv"

# Leitura dos DataFrames com schema
df_estatisticas = spark.read.csv(path_estatisticas, header=True, schema=schema_estatisticas)
df_projetos = spark.read.csv(path_projetos, header=True, schema=schema_projetos)
df_bolsas = spark.read.csv(path_bolsas, header=True, schema=schema_bolsas)
df_disciplinas = spark.read.csv(path_disciplinas, header=True, schema=schema_disciplinas)

In [0]:
# Testando se tá tudo ok
display(df_estatisticas.limit(5))
display(df_projetos.limit(5))
display(df_bolsas.limit(5))
display(df_disciplinas.limit(5))


ano,periodo,curso,municipio,turno,ingressantes,ingressantes_nao_cotistas,ingressantes_cotistas,ativos,concluidos,trancados,cancelados
2025,2,INTERDISCIPLINAR EM CIÊNCIA E TECNOLOGIA,ANGICOS,Matutino e Vespertino,6,6,0,147,0,8,3
2025,2,ENGENHARIA CIVIL,ANGICOS,Matutino e Vespertino,9,9,0,56,0,3,0
2025,2,ENGENHARIA DE PRODUÇÃO,ANGICOS,Matutino e Vespertino,11,11,0,34,0,2,0
2025,2,INTERDISCIPLINAR EM CIÊNCIA E TECNOLOGIA,ANGICOS,Noturno,6,6,0,147,0,8,3
2025,2,INTERDISCIPLINAR EM CIÊNCIA E TECNOLOGIA.,ANGICOS,Noturno,36,30,6,218,0,7,2


codigo,titulo_projeto,linha_de_pesquisa,area_conhecimento_cnpq,unidade,coordenador_docente_interno,coordenador_docente_externo,palavras_chave,data_inicio,data_fim,motivo_finalizacao,status_atual,entidade_financiadora,valor_financiamento
PIH0000003-2024,Modelagem paramétrica e biomimética na Arquitetura,Representação e linguagem de projeto,Planejamento e Projetos da Edificação,DEPARTAMENTO DE CIÊNCIAS SOCIAIS APLICADAS E HUMANAS - PAU DOS FERROS,CAMILA CAVANCANTI RESENDE,null,arquietura paramétrica; biomimética; fabricação digital; projetos sustentáveis,01-set-2024,31-ago-2025,null,PENDENTE DE REL. FINAL,null,null
PID2000037-2025,Pesca e Biologia da albacora bandolim e laje na capturados pela pesca de espinhel e cardume associado em Areia Branca-RN e Natal-RN,Aquicultura e Recursos Pesqueiros,Manejo e Conservação de Recursos Pesqueiros Marinhos,DEPARTAMENTO DE CIÊNCIAS ANIMAIS,HUMBERTO GOMES HAZIN,null,"Biometria, Biologia, Atuns, Cardume associado, Espinhel",25-out-2025,24-out-2030,null,AGUARDANDO AUTORIZAÇÃO DA UNIDADE,null,null
PID2000002-2025,"Potencializando e Monitorando a Efetividade da Pesca através do Sensoriamento Remoto, em Tempo Real  PMEPS",Sensoriamento remoto aplicado a pesca,Recursos Pesqueiros Marinhos,DEPARTAMENTO DE CIÊNCIAS ANIMAIS,HUMBERTO GOMES HAZIN,null,"Sensoriamento remoto, pesca de atuns, monitoramento pesqueiro em tempo real, gestao e manejo",01-mar-2025,31-dez-2030,null,EM EXECUÇÃO,null,null
PID2000019-2020,APLICAÇÃO DE TECNICAS DE SENSORIAMENTO REMOTO NA PESCA DE ATUNS E AFINS NA MODALIDADE DE CARDUMES ASSOCIADOS DA FROTA SEDIADA NO ATLANTICO OESTE EQUATORIAL,null,Recursos Pesqueiros Marinhos,DEPARTAMENTO DE CIÊNCIAS ANIMAIS,HUMBERTO GOMES HAZIN,null,"sensoriamento remoto, cardume associado, pesca sustentavel",06-abr-2020,23-dez-2025,null,EM EXECUÇÃO,null,null
PIA0000037-2017,DISTRIBUIÇÃO DE GRANDES PEIXES PELAGICOS NO ATLÂNTICO SUL E EQUATORIAL E POTENCIAL DE UTILIZAÇÃO DE ESTRATÉGIAS DE MANEJO ESPACIAL COMO INSTRUMENTO DE GESTÃO,Manejo e gerenciamento de recursos pesqueiros,Recursos Pesqueiros Marinhos,(EXTINTO) - DEPARTAMENTO DE CIÊNCIAS ANIMAIS,HUMBERTO GOMES HAZIN,null,"Manejo espacial, Habitat essencial, fechamento de áreas, espinhel de superfície, Modelos aditivos generalizados",12-jun-2017,10-dez-2019,null,PENDENTE DE REL. FINAL,null,null


nomediscente,nomeorientador,denominacao_bolsa,fonte_pagadora,data_inicio_concessao_bolsa,data_fim_concessao_bolsa
ABDIEL JÔNATAS ALVES DA SILVA,MATHEUS DA SILVA MENEZES,IC 2022-2023,Conselho Nacional de Desenvolvimento Científico e Tecnológico,01-set-2022,31-ago-2023
ABDIEL JÔNATAS ALVES DA SILVA,IDALMIR DE SOUZA QUEIROZ JÚNIOR,IC 2024-2025,Universidade Federal Rural do Semi-árido,04-set-2024,02-jan-2025
ABIGAIL ALVES DE SOUSA BARBOSA,FABIO CHAVES NOBRE,IC 2022-2023,Conselho Nacional de Desenvolvimento Científico e Tecnológico,02-set-2022,15-jun-2023
ABIGAIL ALVES DE SOUSA BARBOSA,FABIO CHAVES NOBRE,IC 2021-2022,Conselho Nacional de Desenvolvimento Científico e Tecnológico,01-set-2021,31-ago-2022
ABINOA PRAXEDES FORTE,ANTONIO ALISSON ALENCAR FREITAS,IC 2025-2026,Conselho Nacional de Desenvolvimento Científico e Tecnológico,08-set-2025,31-ago-2026


curso,turno,campi,nome_curriculo,nome_disciplina,obrigatoria,carga_horaria_total_disciplina
ENGENHARIA AGRÍCOLA E AMBIENTAL,Matutino e Vespertino,MOSSORÓ,2013,ATIVIDADES COMPLEMENTARES,null,60
ENGENHARIA AGRÍCOLA E AMBIENTAL,Matutino e Vespertino,MOSSORÓ,2013,ATIVIDADES COMPLEMENTARES,false,150
ENGENHARIA AGRÍCOLA E AMBIENTAL,Matutino e Vespertino,MOSSORÓ,2013,QUIMICA GERAL,true,60
ENGENHARIA AGRÍCOLA E AMBIENTAL,Matutino e Vespertino,MOSSORÓ,2013,ANALISE E EXPRESSAO TEXTUAL (1200536),true,60
ENGENHARIA AGRÍCOLA E AMBIENTAL,Matutino e Vespertino,MOSSORÓ,2013,LABORATORIO DE QUIMICA GERAL,true,30


**3** - Apresente os cinco períodos (ano e período) que tiveram o maior número de ingressantes no curso de ENGENHARIA DE SOFTWARE.

In [0]:
from pyspark.sql.functions import concat_ws, col, sum as _sum

(
    df_estatisticas
    .filter(col("curso") == "ENGENHARIA DE SOFTWARE") # filtrar pelo curso de engenharia de software
    .groupBy(concat_ws(" - ", col("ano"), col("periodo")).alias("ano_periodo")) # concatenação do ano e período, apelidado de ano_periodo, para agrupar pelos dois juntos
    .agg(_sum("ingressantes").alias("total_ingressantes")) # calcula a soma total dos estudantes, apelidado de total_ingressantes
    .orderBy(col("total_ingressantes").desc()) # Ordena os resultados em ordem decrescente pelo total de ingressantes, pra o limite...
    .limit(5) #... já ser os a maior contagem
    .show(truncate=False) # isso é só pra não encurtar as colunas, mostrar melhor
)


+-----------+------------------+
|ano_periodo|total_ingressantes|
+-----------+------------------+
|2025 - 2   |23                |
|2024 - 3   |21                |
|2024 - 1   |15                |
|2023 - 2   |9                 |
|2023 - 1   |8                 |
+-----------+------------------+



**4** - Apresente os 10 cursos da UFERSA com o maior número de ingressantes no ano de 2024.

In [0]:
from pyspark.sql.functions import col, sum as _sum

(
    df_estatisticas
    .filter(col("ano") == 2024) # Filtra apenas os registros referentes ao ano de 2024
    .groupBy("curso") # Agrupa por curso
    .agg(_sum("ingressantes").alias("total_ingressantes")) #Somar todos os ingressantes de cada curso
    .orderBy(col("total_ingressantes").desc()) # Ordena em ordem decrescente pelo total de ingressantes já pros maiores números subirem
    .limit(10) # Seleciona apenas as 10 linhas, que já são os mais cheios
    .show(truncate=False)
)


+--------------------------------------------+------------------+
|curso                                       |total_ingressantes|
+--------------------------------------------+------------------+
|COMPUTAÇÃO                                  |422               |
|INTERDISCIPLINAR EM CIÊNCIA E TECNOLOGIA    |348               |
|MATEMÁTICA                                  |336               |
|INTERDISCIPLINAR EM CIÊNCIA E TECNOLOGIA.   |250               |
|QUÍMICA                                     |150               |
|FÍSICA                                      |145               |
|CIÊNCIA DA COMPUTAÇÃO                       |100               |
|INTERDISCIPLINAR EM TECNOLOGIA DA INFORMAÇÃO|82                |
|ENGENHARIA CIVIL                            |82                |
|AGRONOMIA                                   |64                |
+--------------------------------------------+------------------+



**5** - Apresente o quantitativo de alunos ingressantes, ativos e concluídos por município a cada ano.

In [0]:
from pyspark.sql.functions import col, sum as _sum

(
    df_estatisticas
    .groupBy("ano", "municipio") # Agrupa os dados por ano e município
    .agg( # Calcula a soma de ingressantes, ativos e concluídos por grupo, sendo renomeados pelo alias
        _sum("ingressantes").alias("total_ingressantes"),
        _sum("ativos").alias("total_ativos"),
        _sum("concluidos").alias("total_concluidos")
    )
    .orderBy(col("ano").asc(), col("municipio").asc()) # Ordena o resultado por ano e depois por município
    .show(truncate=False)
)


+----+---------+------------------+------------+----------------+
|ano |municipio|total_ingressantes|total_ativos|total_concluidos|
+----+---------+------------------+------------+----------------+
|199 |MOSSORÓ  |1                 |0           |0               |
|1968|MOSSORÓ  |14                |14          |0               |
|1969|MOSSORÓ  |44                |57          |0               |
|1970|MOSSORÓ  |15                |63          |0               |
|1971|MOSSORÓ  |1                 |57          |0               |
|1972|MOSSORÓ  |1                 |40          |21              |
|1973|MOSSORÓ  |1                 |22          |0               |
|1974|MOSSORÓ  |4                 |10          |0               |
|1975|MOSSORÓ  |3                 |12          |0               |
|1976|MOSSORÓ  |3                 |26          |2               |
|1977|MOSSORÓ  |5                 |30          |5               |
|1978|MOSSORÓ  |10                |39          |3               |
|1979|MOSS

**6** - Apresente os 10 professores com o maior número de coordenações de projetos de pesquisa já finalizados.

In [0]:
from pyspark.sql.functions import col, count

(
    df_projetos
    .filter(col("status_atual") == "FINALIZADO") #Filtra apenas os projetos que já foram finalizados
    .groupBy("coordenador_docente_interno") #Agrupa pelos responsáveis
    .agg(count("*").alias("total_projetos_finalizados")) #Conta o número total de projetos finalizados por coordenador
    .orderBy(col("total_projetos_finalizados").desc()) #Os com mais ficam encima
    .limit(10)
    .show(truncate=False)
)


+-------------------------------+--------------------------+
|coordenador_docente_interno    |total_projetos_finalizados|
+-------------------------------+--------------------------+
|FRANCISCO MILTON MENDES NETO   |24                        |
|RAFAEL OLIVEIRA BATISTA        |12                        |
|DANIEL FREITAS FREIRE MARTINS  |11                        |
|NILDO DA SILVA DIAS            |11                        |
|CARLOS EDUARDO BEZERRA DE MOURA|11                        |
|MARCELO TAVARES GURGEL         |11                        |
|MARCO ANTONIO DIODATO          |10                        |
|GABRIELA SALAMI                |10                        |
|MICHELLY FERNANDES DE MACEDO   |9                         |
|IONÁ SANTOS ARAÚJO HOLANDA     |9                         |
+-------------------------------+--------------------------+



**7** - Apresente os 10 professores com o maior número de orientações de bolsas de iniciação científica.

In [0]:
from pyspark.sql.functions import col, count

(
    df_bolsas
    .groupBy("nomeorientador")
    .agg(count("*").alias("total_orientacoes")) #Conta o número total de orientações por orientador
    .orderBy(col("total_orientacoes").desc())
    .limit(10)
    .show(truncate=False)
)


+-------------------------------------+-----------------+
|nomeorientador                       |total_orientacoes|
+-------------------------------------+-----------------+
|ALEXSANDRA FERNANDES PEREIRA         |32               |
|DANIEL VALADAO SILVA                 |32               |
|CIBELE DOS SANTOS BORGES             |30               |
|GLAUBER HENRIQUE DE SOUSA NUNES      |28               |
|ALEXANDRE RODRIGUES SILVA            |26               |
|HUMBERTO DIONISIO DE ANDRADE         |25               |
|FRANCISCO DE ASSIS DE OLIVEIRA       |24               |
|FRANCISCO SILVESTRE BRILHANTE BEZERRA|22               |
|CAIO AUGUSTO MARTINS AIRES           |22               |
|RAFAEL RODOLFO DE MELO               |22               |
+-------------------------------------+-----------------+



**8** - Apresente a lista de todas as disciplinas do curso de ENGENHARIA DE SOFTWARE.

In [0]:
from pyspark.sql.functions import col, when

(
    df_disciplinas
    .filter(col("curso") == "ENGENHARIA DE SOFTWARE") # Filtro pelo curso de Engenharia de Software
    .select( # Seleciona as colunas relevantes
        "nome_disciplina",
        when(col("obrigatoria") == True, "Obrigatória") # Trocando True e False pelo significado, para facilitar a leitura
        .otherwise("Optativa")
        .alias("tipo_disciplina"), # Renomeando pois a coluna original possui um nome confuso
        "carga_horaria_total_disciplina"
    )    .distinct() # Elimina disciplinas duplicadas
    .orderBy(col("nome_disciplina").asc())
    .show(truncate=False)
)


+------------------------------------------------------+---------------+------------------------------+
|nome_disciplina                                       |tipo_disciplina|carga_horaria_total_disciplina|
+------------------------------------------------------+---------------+------------------------------+
|ADMINISTRACAO E EMPREENDEDORISMO                      |Obrigatória    |60                            |
|ALGEBRA LINEAR (1200260)                              |Obrigatória    |60                            |
|ALGORITMOS                                            |Obrigatória    |60                            |
|ALGORITMOS E ESTRUTURAS DE DADOS I                    |Obrigatória    |60                            |
|ALGORITMOS E ESTRUTURAS DE DADOS II                   |Obrigatória    |60                            |
|ANALISE E EXPRESSAO TEXTUAL (1200536)                 |Obrigatória    |60                            |
|ANÁLISE E PROJETO DE SISTEMAS ORIENTADOS A OBJETOS    |Obrigató

**9** - Apresente os 10 cursos com a maior quantidade de disciplinas não obrigatórias ofertadas no período noturno. Cursos que possuem o mesmo nome, mas outras características diferentes (turno, campi e currículo) devem ser tratados como cursos diferentes.

In [0]:
from pyspark.sql.functions import col, count

(
    df_disciplinas
    .filter(  # Filtra apenas as disciplinas que atendem aos critérios
        (col("obrigatoria") == False) &  # Apenas não obrigatórias no período noturno
        (col("turno") == "NOTURNO")  
    )
    .groupBy( # Agrupa para tratar como diferentes
        "curso",        
        "turno",         
        "campi",       
        "nome_curriculo"      
    )
    .agg(
        count("nome_disciplina").alias("qtd_disciplinas_optativas")  # Conta as disciplinas
    )
    .orderBy(col("qtd_disciplinas_optativas").desc()) 
    .limit(10)  
    .show(truncate=False)
)


+-----+-----+-----+--------------+-------------------------+
|curso|turno|campi|nome_curriculo|qtd_disciplinas_optativas|
+-----+-----+-----+--------------+-------------------------+
+-----+-----+-----+--------------+-------------------------+



**10** - Apresente os 10 cursos com a maior carga horária de disciplinas obrigatórias. Cursos que possuem o mesmo nome, mas outras características diferentes (turno, campi e currículo) devem ser tratados como cursos diferentes.

In [0]:
from pyspark.sql.functions import col, sum as soma # Não sabia que era palavra reservada, pense num aperreio

(
    df_disciplinas
    .filter(col("obrigatoria") == True)  # Filtra apenas disciplinas obrigatórias
    .groupBy(
        "curso",        
        "turno",        
        "campi",       
        "nome_curriculo"
    )
    .agg(
        soma("carga_horaria_total_disciplina").alias("carga_horaria_total_obrigatoria")  
    )
    .orderBy(col("carga_horaria_total_obrigatoria").desc())
    .limit(10)
    .show(truncate=False)
)


+-------------------------------------+---------------------+-------+--------------+-------------------------------+
|curso                                |turno                |campi  |nome_curriculo|carga_horaria_total_obrigatoria|
+-------------------------------------+---------------------+-------+--------------+-------------------------------+
|INTERDISCIPLINAR EM EDUCAÇÃO DO CAMPO|Matutino e Vespertino|MOSSORÓ|2013          |9000                           |
|MEDICINA                             |Matutino e Vespertino|MOSSORÓ|2016          |8536                           |
|INTERDISCIPLINAR EM EDUCAÇÃO DO CAMPO|Matutino e Vespertino|MOSSORÓ|2019          |5940                           |
|MEDICINA VETERINÁRIA                 |Matutino e Vespertino|MOSSORÓ|2004          |4140                           |
|ENGENHARIA FLORESTAL                 |Matutino e Vespertino|MOSSORÓ|2015          |4010                           |
|ENGENHARIA DE PESCA                  |Matutino e Vespertino|MOS